# 🤖 Step 2.1 — Baseline Models (TF-IDF + LogReg / Linear SVM)

**Mục tiêu:**
- Load dữ liệu đã tiền xử lý từ `data/processed/`
- Trích xuất TF-IDF features (word 1-2gram + char 3-5gram)
- Huấn luyện baseline ABSA: mỗi aspect là 1 bài toán phân loại đa lớp
  (`__NONE__` / `positive` / `negative` / `neutral`)
- So sánh **Logistic Regression** vs **Linear SVM** với `class_weight='balanced'`
- Xuất Macro-F1 per aspect + overall, confusion matrix, lưu artifacts

In [ ]:
import sys, os, json, pathlib, subprocess, importlib, warnings
warnings.filterwarnings('ignore')

root = pathlib.Path('..').resolve()
sys.path.insert(0, str(root))

def _ensure(mod, pkg=None):
    pkg = pkg or mod
    try:
        importlib.import_module(mod)
    except ImportError:
        print(f'📦 Cài {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for m, p in [('pandas','pandas'), ('numpy','numpy'), ('sklearn','scikit-learn'),
             ('pyarrow','pyarrow'), ('matplotlib','matplotlib'), ('seaborn','seaborn'),
             ('joblib','joblib')]:
    _ensure(m, p)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (classification_report, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)

sns.set_theme(style='whitegrid')
print('✅ Imports OK')

## 1. Load processed data + metadata

In [ ]:
processed_dir = pathlib.Path('../data/processed')
assert processed_dir.exists(), f"❌ Chưa có {processed_dir} — chạy 02_Preprocessing.ipynb trước."

train = pd.read_parquet(processed_dir / 'train.parquet')
val   = pd.read_parquet(processed_dir / 'val.parquet')
test  = pd.read_parquet(processed_dir / 'test.parquet')

with open(processed_dir / 'metadata.json', encoding='utf-8') as fh:
    meta = json.load(fh)
with open(processed_dir / 'class_weights.json', encoding='utf-8') as fh:
    class_weights = json.load(fh)

review_col  = meta['review_col']
aspect_cols = meta['aspect_cols']
format_type = meta['format_type']

print(f"📊 Train/Val/Test: {len(train):,} / {len(val):,} / {len(test):,}")
print(f"📝 Review col    : {review_col}")
print(f"🏷️  Aspects ({len(aspect_cols)}): {aspect_cols[:6]}{'...' if len(aspect_cols)>6 else ''}")
print(f"🗂️  Format       : {format_type}")

# Text input: ưu tiên text_tokenized (có tokenize VN), fallback text_clean
text_col = 'text_tokenized' if 'text_tokenized' in train.columns else 'text_clean'
print(f"📄 Text col      : {text_col}")
train.head(2)

## 2. Chuẩn hóa labels cho multi-output classification

Với mỗi aspect → 1 bài toán 4-class: `__NONE__` / `positive` / `negative` / `neutral`.
NaN được coi là `__NONE__` (không đề cập aspect này).

In [ ]:
NONE_LABEL = '__NONE__'

def build_labels(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    Y = df[cols].copy()
    if format_type == 'dense':
        # Dense: các giá trị 'positive'/'negative'/'neutral'/NaN
        Y = Y.fillna(NONE_LABEL).astype(str)
    else:
        # Sparse: có thể là NaN hoặc string label
        Y = Y.fillna(NONE_LABEL).astype(str)
    return Y

Y_train = build_labels(train, aspect_cols)
Y_val   = build_labels(val, aspect_cols)
Y_test  = build_labels(test, aspect_cols)

X_train = train[text_col].fillna('').astype(str).values
X_val   = val[text_col].fillna('').astype(str).values
X_test  = test[text_col].fillna('').astype(str).values

print("Phân phối label (train) — 3 aspect đầu:")
for a in aspect_cols[:3]:
    print(f"\n  {a}:")
    print(Y_train[a].value_counts().to_string())

## 3. TF-IDF Vectorizer (word + char n-gram)

In [ ]:
vectorizer = FeatureUnion([
    ('word', TfidfVectorizer(
        analyzer='word', ngram_range=(1, 2),
        min_df=2, max_df=0.95, max_features=30_000, sublinear_tf=True,
    )),
    ('char', TfidfVectorizer(
        analyzer='char_wb', ngram_range=(3, 5),
        min_df=2, max_df=0.95, max_features=30_000, sublinear_tf=True,
    )),
])

print('🔧 Fitting TF-IDF trên train ...')
Xtr = vectorizer.fit_transform(X_train)
Xvl = vectorizer.transform(X_val)
Xte = vectorizer.transform(X_test)

print(f"  Train matrix: {Xtr.shape}")
print(f"  Val   matrix: {Xvl.shape}")
print(f"  Test  matrix: {Xte.shape}")

## 4. Train baseline: 1 classifier / aspect

In [ ]:
def train_per_aspect(model_factory, Xtr, Ytr, Xvl, Yvl):
    """Train 1 classifier cho mỗi aspect, trả dict[aspect] -> (model, val_preds)."""
    results = {}
    for asp in aspect_cols:
        y_tr = Ytr[asp].values
        y_vl = Yvl[asp].values
        # Cần ít nhất 2 class để train
        if len(set(y_tr)) < 2:
            results[asp] = (None, np.array([y_tr[0]] * len(y_vl)))
            continue
        clf = model_factory()
        clf.fit(Xtr, y_tr)
        pred = clf.predict(Xvl)
        results[asp] = (clf, pred)
    return results

def eval_multi(Y_true: pd.DataFrame, preds: dict) -> pd.DataFrame:
    rows = []
    for asp in aspect_cols:
        y = Y_true[asp].values
        p = preds[asp]
        macro = f1_score(y, p, average='macro', zero_division=0)
        micro = f1_score(y, p, average='micro', zero_division=0)
        # F1 chỉ trên class có opinion (loại __NONE__)
        mask = y != NONE_LABEL
        if mask.sum() > 0:
            f1_op = f1_score(y[mask], p[mask], average='macro', zero_division=0)
        else:
            f1_op = np.nan
        rows.append({'aspect': asp, 'macro_f1': macro,
                     'micro_f1': micro, 'f1_opinion_only': f1_op})
    return pd.DataFrame(rows).set_index('aspect')

models_lr = train_per_aspect(
    lambda: LogisticRegression(max_iter=2000, class_weight='balanced',
                               C=1.0, solver='liblinear', n_jobs=None),
    Xtr, Y_train, Xvl, Y_val,
)
print('✅ LogReg trained')

models_svm = train_per_aspect(
    lambda: LinearSVC(C=1.0, class_weight='balanced'),
    Xtr, Y_train, Xvl, Y_val,
)
print('✅ LinearSVC trained')

## 5. Đánh giá trên Val

In [ ]:
preds_lr_val  = {a: m[1] for a, m in models_lr.items()}
preds_svm_val = {a: m[1] for a, m in models_svm.items()}

eval_lr  = eval_multi(Y_val, preds_lr_val).add_suffix('_lr')
eval_svm = eval_multi(Y_val, preds_svm_val).add_suffix('_svm')
eval_val = pd.concat([eval_lr, eval_svm], axis=1).round(4)

print('=== VAL Macro-F1 per aspect ===')
display(eval_val)

overall = pd.DataFrame({
    'LogReg':    eval_lr.mean().rename(lambda s: s.replace('_lr', '')),
    'LinearSVM': eval_svm.mean().rename(lambda s: s.replace('_svm', '')),
}).round(4)
print('\n=== VAL Overall (mean across aspects) ===')
display(overall)

In [ ]:
# Biểu đồ so sánh Macro-F1 per aspect
fig, ax = plt.subplots(figsize=(max(8, 0.45 * len(aspect_cols)), 4.5))
x = np.arange(len(aspect_cols))
w = 0.4
ax.bar(x - w/2, eval_val['macro_f1_lr'],  w, label='LogReg',    color='#42A5F5')
ax.bar(x + w/2, eval_val['macro_f1_svm'], w, label='LinearSVM', color='#EF5350')
ax.set_xticks(x)
ax.set_xticklabels(aspect_cols, rotation=45, ha='right')
ax.set_ylabel('Macro-F1 (val)')
ax.set_title('Baseline Macro-F1 per Aspect')
ax.legend()
plt.tight_layout()

out_dir = pathlib.Path('../outputs/baseline')
out_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(out_dir / 'val_macro_f1_per_aspect.png', dpi=120)
plt.show()

## 6. Chọn model tốt hơn → đánh giá trên Test

In [ ]:
best_name = 'LogReg' if overall.loc['macro_f1', 'LogReg'] >= overall.loc['macro_f1', 'LinearSVM'] else 'LinearSVM'
best_models = models_lr if best_name == 'LogReg' else models_svm
print(f"🏆 Best on val: {best_name}")

# Predict test
preds_test = {}
for asp in aspect_cols:
    clf, _ = best_models[asp]
    if clf is None:
        preds_test[asp] = np.array([Y_train[asp].iloc[0]] * len(test))
    else:
        preds_test[asp] = clf.predict(Xte)

eval_test = eval_multi(Y_test, preds_test).round(4)
print('\n=== TEST Macro-F1 per aspect ===')
display(eval_test)

print('\n=== TEST Overall ===')
overall_test = eval_test.mean().round(4)
display(overall_test.to_frame('value'))

In [ ]:
# Classification report chi tiết cho 3 aspect có sample nhiều nhất
top_aspects = Y_train.apply(lambda s: (s != NONE_LABEL).sum()).sort_values(ascending=False).head(3).index.tolist()
for asp in top_aspects:
    print(f"\n━━━ {asp} ━━━")
    print(classification_report(Y_test[asp].values, preds_test[asp], zero_division=0))

In [ ]:
# Confusion matrix cho aspect đầu tiên (có nhiều sample nhất)
asp = top_aspects[0]
labels_order = [NONE_LABEL, 'positive', 'negative', 'neutral']
labels_order = [l for l in labels_order if l in set(Y_test[asp]) | set(preds_test[asp])]

cm = confusion_matrix(Y_test[asp].values, preds_test[asp], labels=labels_order)
fig, ax = plt.subplots(figsize=(5, 4.5))
disp = ConfusionMatrixDisplay(cm, display_labels=labels_order)
disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
ax.set_title(f'Confusion Matrix — {asp} ({best_name})')
plt.tight_layout()
plt.savefig(out_dir / f'cm_{asp}.png', dpi=120)
plt.show()

## 7. Lưu artifacts

In [ ]:
# Lưu vectorizer + models + metrics
joblib.dump(vectorizer, out_dir / 'tfidf_vectorizer.joblib')

model_bundle = {asp: m for asp, (m, _) in best_models.items() if m is not None}
joblib.dump(model_bundle, out_dir / f'baseline_{best_name.lower()}.joblib')

eval_val.to_csv(out_dir / 'val_metrics.csv')
eval_test.to_csv(out_dir / 'test_metrics.csv')

results = {
    'best_model':    best_name,
    'text_col':      text_col,
    'n_aspects':     len(aspect_cols),
    'val_overall':   overall.to_dict(),
    'test_overall':  overall_test.to_dict(),
    'tfidf_shape':   list(Xtr.shape),
}
with open(out_dir / 'summary.json', 'w', encoding='utf-8') as fh:
    json.dump(results, fh, ensure_ascii=False, indent=2, default=str)

print(f"✅ Saved to {out_dir.resolve()}")
for p in sorted(out_dir.iterdir()):
    print(f"   • {p.name}")

---
## ✅ Checklist Step 2.1

- [x] Load `train/val/test.parquet` + metadata + class_weights
- [x] TF-IDF word(1-2) + char(3-5), sublinear_tf
- [x] Multi-output per aspect (4-class: NONE/pos/neg/neu)
- [x] LogReg & LinearSVM với `class_weight='balanced'`
- [x] Macro-F1 per aspect + overall (val + test)
- [x] Confusion matrix + classification report
- [x] Lưu vectorizer, models, metrics, summary vào `outputs/baseline/`

**Bước tiếp theo:** Step 2.2 — Neural baselines (BiLSTM + fastText embedding) hoặc Step 3 — PhoBERT fine-tuning.